In [ ]:
# Copy Default Storage

from datetime import datetime
from getpass import getpass
import os

rdm_url=None
idp_name_1=None
idp_username_1=None
idp_password_1=None

jupyterhub_username = None
jupyterhub_password = None

default_result_path = None
close_on_fail = False
transition_timeout = 60 * 1000

project_name = None

binderhub_post_build_script = '''#!/bin/bash
date > ~/uptime'''
binderhub_dockerfile_option = '!SINGLESPEED!'
binderhub_launch_timeout = 30 * 60 * 1000 # 30 minutes
dockerfile_custom_script = """FROM gcr.io/nii-ap-ops/base/datascience-notebook:main-lab4.x

COPY --chown=$NB_UID:$NB_GID . .
"""
binderhub_test_filename = 'grdm_test_file.txt'
binderhub_test_file_content = 'GRDM_FILE_SYNC_TEST'


In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if jupyterhub_username is None:
    idp_username_1 = input(prompt=f'Username for {jupyterhub_username}')
if jupyterhub_password is None:
    idp_password_1 = getpass(prompt=f'Password for {jupyterhub_password}@JupyterHub')
if project_name is None:
    project_name = datetime.now().strftime('TEST-BINDERHUB-%Y%m%d%H%M')

project_url = None
project_created = False
environment_name = None


# BinderHubアドオン デフォルトストレージのコピー制御

- サブシステム名: アドオン
- ページ/アドオン: BinderHub
- 機能分類: アドオン操作
- シナリオ名: デフォルトストレージのコピー制御
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: Local RDM + JupyterHub, Local RDMは全てプロフィールを埋めていること / JupyterHubはサーバが追加で3個作れる状態であること)
- 事前条件: なし

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


In [ ]:
import asyncio

async def run_lab_command(page, command, expected_substring, run_wait_time=3000):
    editor = page.locator('div.jp-CodeCell div.cm-content').nth(0)
    await expect(editor).to_be_visible(timeout=transition_timeout)
    await editor.click()
    await editor.fill(command)

    run_button = page.locator('//jp-button[@data-command = "notebook:run-cell-and-select-next"]')
    await expect(run_button).to_be_visible(timeout=transition_timeout)
    await run_button.click()
    await asyncio.sleep(run_wait_time / 1000)

    output = page.locator(f'//*[contains(@class, "jp-OutputArea")]//*[text()[contains(., "{expected_substring}")]]')
    await expect(output).to_be_visible(timeout=transition_timeout)


## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

Local RDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url, wait_until="networkidle")
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

await run_pw(_step)


## IdPを利用し、既存ユーザー1としてログインする

Local RDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## BinderHubアドオンを有効化する

アドオン利用規約の確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.enable_addon(page, 'GakuNin Federated Computing Services (Jupyter)', transition_timeout=transition_timeout)

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

BinderHubアドオンページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    # Login to JupyterHub
    jupyterhub_signin_button = page.locator('//*[@id = "login_submit"]')
    await expect(jupyterhub_signin_button).to_be_visible(timeout=transition_timeout)
    await page.locator('//*[@id = "username_input"]').fill(jupyterhub_username)
    await page.locator('//*[@id = "password_input"]').fill(jupyterhub_password)
    await jupyterhub_signin_button.click()
    # Authorize BinderHub
    authorize_button = page.locator('//*[@value = "Authorize"]')
    await expect(authorize_button).to_be_visible(timeout=transition_timeout)
    await authorize_button.click()
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「ファイル」をクリックする

フォルダ「.binder」が作成されていること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-node-navbar-link = "files"]').click()
    binder_folder = page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-plus")]')
    await expect(binder_folder).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## プロジェクトのストレージにテストファイルをアップロードする

ファイルがアップロードされ、ファイル一覧に表示されること

In [ ]:
async def _step(page):
    # テストファイルを作成
    test_filepath = os.path.join(work_dir, binderhub_test_filename)
    with open(test_filepath, 'w') as f:
        f.write(binderhub_test_file_content)

    # NII Storageを選択してアップロード
    await grdm.get_select_storage_title_locator(page, 'NII Storage').click()
    await grdm.upload_file(page, test_filepath)
    await expect(grdm.get_select_file_title_locator(page, binderhub_test_filename)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

BinderHubアドオンページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

環境の起動を待つ

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup', timeout=binderhub_launch_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    return popup

await run_pw(_step)


## JupyterLabが表示されていることを確認する

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@class = "jp-LauncherCard" and @title = "Python 3 (ipykernel)" and @data-category = "Notebook"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新規Notebookを作成する

In [ ]:
async def _step(page):
    launcher_label = 'Python 3 (ipykernel)'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトのファイルがコピーされていることを確認する

テストファイルの内容が出力されること

In [ ]:
async def _step(page):
    await run_lab_command(page, f'!cat {binderhub_test_filename}', binderhub_test_file_content)

await run_pw(_step)

## Notebookを閉じる

In [ ]:
async def _step(page):
    close_icon = page.locator('//*[contains(@class, "lm-mod-current")]//*[contains(@class, "lm-TabBar-tabCloseIcon")]')
    await expect(close_icon).to_be_visible(timeout=transition_timeout)
    await close_icon.click()

    discard_button = page.locator('//div[text() = "Discard"]')
    await expect(discard_button).to_be_visible(timeout=transition_timeout)
    await discard_button.click()

await run_pw(_step)

## JupyterLabウィンドウを閉じる

In [ ]:
await close_latest_page()

async def _step(page):
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「デフォルトストレージの内容をコピーする」チェックボックスをオフにする

チェックボックスがオフになること

In [ ]:
async def _step(page):
    checkbox = page.locator('//*[@data-test-copy-default-storage]')
    await expect(checkbox).to_be_checked(timeout=transition_timeout)
    await checkbox.click()

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

環境の起動を待つ

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup', timeout=binderhub_launch_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    return popup

await run_pw(_step)


## JupyterLabが表示されていることを確認する

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@class = "jp-LauncherCard" and @title = "Python 3 (ipykernel)" and @data-category = "Notebook"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新規Notebookを作成する

In [ ]:
async def _step(page):
    launcher_label = 'Python 3 (ipykernel)'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## プロジェクトのファイルがコピーされていないことを確認する

テストファイルが存在せず、エラーが出力されること

In [ ]:
async def _step(page):
    await run_lab_command(page, f'!cat {binderhub_test_filename}', 'No such file or directory')

await run_pw(_step)


## JupyterLabウィンドウを閉じる

In [ ]:
await close_latest_page()

async def _step(page):
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「デフォルトストレージの内容をコピーする」チェックボックスをオンに戻す

チェックボックスがオンになること

In [ ]:
async def _step(page):
    checkbox = page.locator('//*[@data-test-copy-default-storage]')
    await expect(checkbox).not_to_be_checked(timeout=transition_timeout)
    await checkbox.click()

await run_pw(_step)


## 基本イメージの「変更」をクリックする

Dockerfileオプションが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-image-change]').click()
    await expect(page.locator(f'//*[@data-test-image-selection = "{binderhub_dockerfile_option}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Dockerfileを用いたカスタムイメージ」をクリックする

「Dockerfile」エディターが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-image-selection = "{binderhub_dockerfile_option}"]').click()
    await expect(page.locator('//h3[text() = "Dockerfile"]/../..//textarea')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Dockerfile」エディターにスクリプトを入力する

入力内容が表示されること

In [ ]:
async def _step(page):
    textarea = page.locator('//h3[text() = "Dockerfile"]/../..//textarea')
    await textarea.fill(dockerfile_custom_script)
    await textarea.press('Tab')

await run_pw(_step)


## 「保存」をクリックする

エラーメッセージが表示されないこと

In [ ]:
import asyncio

async def _step(page):
    await page.locator('//h3[text() = "Dockerfile"]/../..//button[text() = "保存"]').click()
    await asyncio.sleep(10)

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

環境の起動を待つ

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup', timeout=binderhub_launch_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    return popup

await run_pw(_step)


## JupyterLabが表示されていることを確認する

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@class = "jp-LauncherCard" and @title = "Python 3 (ipykernel)" and @data-category = "Notebook"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトのファイルがコピーされていることを確認する

新規Notebookを作成し、テストファイルの内容が出力されること

In [ ]:
async def _step(page):
    launcher_label = 'Python 3 (ipykernel)'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    # Execution Indicator は、 ... でCollapseされ表示されない可能性がある
    try:
        await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

        await asyncio.sleep(5)
    except:
        # Warning
        traceback.print_exc()

    await run_lab_command(page, f'!cat {binderhub_test_filename}', binderhub_test_file_content)

await run_pw(_step)

## JupyterLabウィンドウを閉じる

In [ ]:
await close_latest_page()

async def _step(page):
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


終了処理を実施する。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}